# Allen Visual Learning Project EDA

This notebook explores the Allen Visual Learning dataset for our project on how inhibitory neuron types relate to neural activity and visual learning.

The analysis focuses on:
- session structure and learning stages
- calcium activity
- behavior and visual stimuli
- transcriptomic cell-type labels
- initial event-aligned and cell-type comparisons

## 1. Environment

Dependencies are defined in the project environment. This cell can be used if packages are missing.

## 2. Imports and configuration

Set the local data directory and choose an example mouse and session for the initial analysis.

In [ ]:
from pathlib import Path
from datetime import datetime
import glob
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
import pynwb

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

CODE_OCEAN_ROOT = Path("/data")
LOCAL_ROOT = Path("./allen_visual_learning_data")

if (CODE_OCEAN_ROOT / "Visual-Learning-SWDB").exists():
    DATA_ROOT = CODE_OCEAN_ROOT
    IN_CODE_OCEAN = True
else:
    DATA_ROOT = LOCAL_ROOT
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    IN_CODE_OCEAN = False

MOUSE = 800995
SESSION_TYPE = "OPHYS_1_images_A"

print("Data root:", DATA_ROOT.resolve())
print("Code Ocean assets detected:", IN_CODE_OCEAN)

## 3. Session metadata

Query the Allen metadata service to identify the available Visual Learning sessions and organize them by mouse, session type, and learning stage.

In [ ]:
from aind_data_access_api.document_db import MetadataDbClient

VISUAL_LEARNING_MICE = ["782149", "790322", "788406", "800792", "800995", "804363"]

client = MetadataDbClient(
    host="api.allenneuraldynamics.org",
    version="v2",
    database="metadata_index",
    collection="data_assets",
)

aggregate = [
    {
        "$match": {
            "data_description.subject_id": {"$in": VISUAL_LEARNING_MICE},
            "data_description.modalities.abbreviation": "pophys",
            "acquisition.acquisition_type": {"$exists": True, "$ne": "CENTER_MOUSEMOTION"},
        }
    },
    {
        "$project": {
            "name": 1,
            "subject_id": "$data_description.subject_id",
            "project_name": "$data_description.project_name",
            "acquisition_type": "$acquisition.acquisition_type",
            "session_start_time": "$acquisition.acquisition_start_time",
            "rig": "$acquisition.instrument_id",
            "planes": {
                "$reduce": {
                    "input": {
                        "$reduce": {
                            "input": "$acquisition.data_streams",
                            "initialValue": [],
                            "in": {"$concatArrays": ["$$value", {"$ifNull": ["$$this.configurations", []]}]},
                        }
                    },
                    "initialValue": [],
                    "in": {
                        "$concatArrays": [
                            "$$value",
                            {
                                "$reduce": {
                                    "input": {"$ifNull": ["$$this.images", []]},
                                    "initialValue": [],
                                    "in": {"$concatArrays": ["$$value", {"$ifNull": ["$$this.planes", []]}]},
                                }
                            },
                        ]
                    },
                }
            },
        }
    },
    {
        "$project": {
            "name": 1,
            "subject_id": 1,
            "project_name": 1,
            "acquisition_type": 1,
            "session_start_time": 1,
            "rig": 1,
            "n_planes": {"$size": "$planes"},
            "plane_indices": "$planes.plane_index",
            "imaging_depths": "$planes.depth",
            "targeted_structures": "$planes.targeted_structure.acronym",
        }
    },
    {"$match": {"n_planes": 8}},
]

records = client.aggregate_docdb_records(pipeline=aggregate)
sessions = pd.DataFrame(records)

processed_pattern = (
    r"^multiplane-ophys_\d+_\d{4}-\d{2}-\d{2}_[\d-]+"
    r"_processed_\d{4}-\d{2}-\d{2}_[\d-]+$"
)

sessions = sessions[sessions["name"].str.match(processed_pattern, na=False)].copy()
sessions["session_id"] = sessions["name"].str.extract(
    r"^(multiplane-ophys_\d+_\d{4}-\d{2}-\d{2}_[\d-]+)_processed_"
)
sessions["processed_stamp"] = sessions["name"].str.extract(
    r"_processed_(\d{4}-\d{2}-\d{2}_[\d-]+)$"
)

sessions = (
    sessions.sort_values("processed_stamp")
    .drop_duplicates("session_id", keep="last")
    .copy()
)

sessions["session_date"] = pd.to_datetime(sessions["session_start_time"]).dt.date
sessions["session_type"] = sessions["acquisition_type"]
sessions["stage"] = sessions["session_type"].str.extract(r"^(TRAINING_\d|OPHYS_\d|STAGE_\d)")
sessions["image_set"] = sessions["session_type"].str.extract(r"_images_([AB])")

sessions = sessions.sort_values(["subject_id", "session_date"]).reset_index(drop=True)
sessions["session_number"] = sessions.groupby("subject_id").cumcount() + 1

def ordered_plane_info(row):
    triples = sorted(
        zip(row["plane_indices"], row["targeted_structures"], row["imaging_depths"]),
        key=lambda x: x[0],
    )
    names = [f"{structure}_{idx}" for idx, structure, depth in triples]
    depths = [depth for idx, structure, depth in triples]
    return names, depths

plane_info = sessions.apply(ordered_plane_info, axis=1)
sessions["plane_names"] = [x[0] for x in plane_info]
sessions["imaging_depths"] = [x[1] for x in plane_info]

print("Unique sessions:", len(sessions))
print("Mice:", sessions["subject_id"].nunique())
display(sessions[["subject_id", "session_number", "session_date", "session_type", "stage", "image_set", "n_planes"]].head(12))

## 4. Session overview

Summarize the available sessions across mice and learning stages.

In [ ]:
session_counts = sessions.groupby(["subject_id", "stage"]).size().unstack(fill_value=0)
display(session_counts)

fig, ax = plt.subplots(figsize=(10, 4))
session_counts.plot(kind="bar", stacked=True, ax=ax)
ax.set_xlabel("mouse")
ax.set_ylabel("number of sessions")
ax.set_title("Visual Learning sessions by mouse and stage")
plt.tight_layout()
plt.show()

In [ ]:
mouse_sessions = sessions[sessions["subject_id"].astype(str) == str(MOUSE)].sort_values("session_number").copy()
display(mouse_sessions[["session_number", "session_date", "session_type", "stage", "image_set"]])

fig, ax = plt.subplots(figsize=(12, 4))
ax.scatter(mouse_sessions["session_number"], np.zeros(len(mouse_sessions)), s=70)
for _, row in mouse_sessions.iterrows():
    ax.text(row["session_number"], 0.02, row["stage"], rotation=60, ha="left", va="bottom", fontsize=8)
ax.set_yticks([])
ax.set_xlabel("session number")
ax.set_title(f"Learning timeline for mouse {MOUSE}")
plt.tight_layout()
plt.show()

## 5. Select a session

Use one familiar-image session as a representative example for the initial EDA.

In [ ]:
matches = mouse_sessions[mouse_sessions["session_type"] == SESSION_TYPE]
if matches.empty:
    print("Requested session type not found. Available session types:")
    display(mouse_sessions["session_type"].value_counts())
    raise ValueError(f"No {SESSION_TYPE!r} session for mouse {MOUSE}")

session = matches.iloc[0]
session

## 6. Load the selected session

The physiology and behavioral data are stored in NWB format. The notebook works with one session at a time.

If the selected session is not already stored locally, it is downloaded from the Allen public data bucket.

In [ ]:
import s3fs

DATASET_DIR = DATA_ROOT / "Visual-Learning-SWDB"
DATASET_DIR.mkdir(parents=True, exist_ok=True)

session_name = session["name"]
selected_session_dir = DATASET_DIR / session_name

if not selected_session_dir.exists():
    fs = s3fs.S3FileSystem(anon=True)

    remote_candidates = [
        f"aind-open-data/{session_name}",
        f"aind-open-data/multiplane-ophys/{session_name}",
    ]

    remote_path = next(
        (path for path in remote_candidates if fs.exists(path)),
        None,
    )

    if remote_path is None:
        raise FileNotFoundError(
            f"Could not find session {session_name} in the Allen public data bucket."
        )

    print(f"Downloading {session_name}")
    fs.get(
        remote_path + "/*",
        str(selected_session_dir),
        recursive=True,
    )

print("Session directory:", selected_session_dir)


## 7. Open the NWB data

NWB (Neurodata Without Borders) is the standardized format used to store the session's neural activity, behavior, stimuli, and metadata.

In [ ]:
nwb_paths = [
    p for p in selected_session_dir.rglob("*")
    if "nwb" in p.name.lower() and not p.name.lower().endswith(".json")
]

if not nwb_paths:
    raise FileNotFoundError("No NWB data found for the selected session.")

nwb_path = nwb_paths[0]
print("Opening:", nwb_path)

nwb = pynwb.read_nwb(str(nwb_path))

print("Session ID:", nwb.session_id)
print("Processing modules:", list(nwb.processing.keys()))
print("Interval tables:", list(nwb.intervals.keys()))
print("Event tables:", list(nwb.events.keys()) if nwb.events else [])


## 8. Physiology overview

Summarize the imaging planes, depths, ROI counts, recording duration, and sampling rate.

In [ ]:
plane_names = [name for name in nwb.processing.keys() if name.startswith("VIS")]
depth_of_plane = dict(zip(session["plane_names"], session["imaging_depths"]))

plane_summary = []
for plane in plane_names:
    dff_series = nwb.processing[plane]["dff_timeseries"]["dff_timeseries"]
    timestamps = np.asarray(dff_series.timestamps[:])
    roi_table = nwb.processing[plane]["image_segmentation"].plane_segmentations["roi_table"].to_dataframe()
    plane_summary.append({
        "plane": plane,
        "depth_um": depth_of_plane.get(plane, np.nan),
        "n_frames": dff_series.data.shape[0],
        "n_rois": dff_series.data.shape[1],
        "n_soma": int(roi_table["is_soma"].sum()) if "is_soma" in roi_table else np.nan,
        "frame_rate_hz": 1 / np.median(np.diff(timestamps)),
        "duration_min": (timestamps[-1] - timestamps[0]) / 60,
    })

plane_summary = pd.DataFrame(plane_summary).sort_values("depth_um")
display(plane_summary)
print("Total ROIs:", plane_summary["n_rois"].sum())
print("Total soma ROIs:", plane_summary["n_soma"].sum())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(plane_summary["depth_um"].astype(str), plane_summary["n_rois"])
ax.set_xlabel("imaging depth (µm)")
ax.set_ylabel("number of ROIs")
ax.set_title("ROIs by imaging depth")
plt.tight_layout()
plt.show()

## 9. Calcium activity

Inspect ΔF/F activity from one imaging plane to understand the structure and range of the neural signals.

In [ ]:
PLANE = plane_summary.iloc[len(plane_summary) // 2]["plane"]

dff_series = nwb.processing[PLANE]["dff_timeseries"]["dff_timeseries"]
dff = np.asarray(dff_series.data[:])
timestamps = np.asarray(dff_series.timestamps[:])

good_rois = ~np.isnan(dff).all(axis=0)
dff = dff[:, good_rois]

print("Plane:", PLANE)
print("dF/F shape:", dff.shape)
print("Time span (min):", (timestamps[-1] - timestamps[0]) / 60)
print("Median dF/F:", np.nanmedian(dff))
print("99th percentile dF/F:", np.nanpercentile(dff, 99))

In [ ]:
mid = timestamps[len(timestamps) // 2]
window = (timestamps >= mid - 30) & (timestamps <= mid + 30)

peak_scores = np.nanpercentile(dff, 99, axis=0)
example_cells = np.argsort(peak_scores)[-5:]

fig, ax = plt.subplots(figsize=(12, 5))
offset = 0
for idx in example_cells:
    trace = dff[window, idx]
    trace = trace - np.nanmedian(trace)
    ax.plot(timestamps[window], trace + offset, lw=1)
    offset += np.nanpercentile(np.abs(trace), 95) + 0.2

ax.set_xlabel("time (s)")
ax.set_ylabel("offset ΔF/F")
ax.set_title(f"Example calcium traces — {PLANE}")
plt.tight_layout()
plt.show()

In [ ]:
n_heatmap = min(50, dff.shape[1])
selected = np.linspace(0, dff.shape[1] - 1, n_heatmap, dtype=int)
heat = dff[window][:, selected].T
row_center = np.nanmedian(heat, axis=1, keepdims=True)
row_scale = np.nanstd(heat, axis=1, keepdims=True)
row_scale[row_scale == 0] = 1
heat_z = (heat - row_center) / row_scale

fig, ax = plt.subplots(figsize=(12, 7))
im = ax.imshow(
    heat_z,
    aspect="auto",
    interpolation="nearest",
    extent=[timestamps[window][0], timestamps[window][-1], n_heatmap, 0],
)
ax.set_xlabel("time (s)")
ax.set_ylabel("sampled neurons")
ax.set_title(f"Calcium activity heatmap — {PLANE}")
fig.colorbar(im, ax=ax, label="within-neuron z-score")
plt.tight_layout()
plt.show()

## 10. Behavior

Explore running speed and trial outcomes from the selected session.

In [ ]:
if "running" in nwb.processing and "speed" in nwb.processing["running"].data_interfaces:
    speed_series = nwb.processing["running"]["speed"]
    speed = np.asarray(speed_series.data[:])
    speed_t = np.asarray(speed_series.timestamps[:])

    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(speed_t, speed, lw=0.5)
    ax.set_xlabel("time (s)")
    ax.set_ylabel("running speed (cm/s)")
    ax.set_title("Running behavior")
    plt.tight_layout()
    plt.show()

    print("Median running speed:", np.nanmedian(speed))
    print("95th percentile running speed:", np.nanpercentile(speed, 95))
else:
    print("Running speed interface not found.")

In [ ]:
if "trials" in nwb.intervals:
    trials = nwb.intervals["trials"].to_dataframe()
    print("Trials:", len(trials))
    display(trials.head())

    outcome_columns = [
        c for c in ["hit", "miss", "false_alarm", "correct_reject", "aborted", "auto_rewarded"]
        if c in trials.columns
    ]
    if outcome_columns:
        outcome_counts = pd.Series({
            c: int(pd.to_numeric(trials[c], errors="coerce").fillna(0).sum())
            for c in outcome_columns
        })
        display(outcome_counts.to_frame("count"))

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(outcome_counts.index, outcome_counts.values)
        ax.set_ylabel("number of trials")
        ax.set_title("Trial outcomes")
        ax.tick_params(axis="x", rotation=35)
        plt.tight_layout()
        plt.show()
else:
    print("This session has no trials table.")

## 11. Visual stimuli

Summarize image presentations, image changes, omissions, and image identities.

In [ ]:
stim = nwb.intervals["stimulus_presentations"].to_dataframe()

print("Stimulus presentations:", len(stim))
print("Columns:")
print(stim.columns.tolist())

if "is_change" in stim:
    print("Image changes:", int(stim["is_change"].fillna(False).sum()))
if "omitted" in stim:
    print("Omitted flashes:", int(stim["omitted"].fillna(False).sum()))
if "image_name" in stim:
    image_counts = stim["image_name"].value_counts()
    display(image_counts)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(image_counts.index.astype(str), image_counts.values)
    ax.set_ylabel("presentations")
    ax.set_xlabel("image")
    ax.set_title("Stimulus presentation counts")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()

## 12. Image-change aligned activity

Align calcium activity around image-change events as an initial view of task-related population responses.

In [ ]:
def event_aligned_mean(data, time, event_times, pre=1.0, post=2.0, dt=0.1):
    grid = np.arange(-pre, post + 1e-9, dt)
    aligned = []
    for event_time in event_times:
        sample_times = event_time + grid
        if sample_times[0] < time[0] or sample_times[-1] > time[-1]:
            continue
        event_matrix = np.empty((len(grid), data.shape[1]), dtype=float)
        for cell in range(data.shape[1]):
            event_matrix[:, cell] = np.interp(sample_times, time, data[:, cell], left=np.nan, right=np.nan)
        aligned.append(event_matrix)
    if not aligned:
        return grid, None
    return grid, np.stack(aligned, axis=0)

if "is_change" in stim.columns:
    change_rows = stim[stim["is_change"].fillna(False)]
    change_times = change_rows["start_time"].to_numpy(dtype=float) if "start_time" in change_rows.columns else change_rows.index.to_numpy(dtype=float)

    rel_t, aligned = event_aligned_mean(dff, timestamps, change_times, pre=1.0, post=2.0, dt=0.1)
    if aligned is not None:
        population_response = np.nanmean(aligned, axis=(0, 2))
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(rel_t, population_response, lw=2)
        ax.axvline(0, linestyle="--")
        ax.set_xlabel("time from image change (s)")
        ax.set_ylabel("mean ΔF/F")
        ax.set_title(f"Population response to image changes — {PLANE}")
        plt.tight_layout()
        plt.show()
        print("Aligned events:", aligned.shape[0])
else:
    print("No is_change column in this session.")

## 13. Get transcriptomic cell type data